# Mount drive

In [ ]:
%pip install -U  transformers datasets peft accelerate huggingface_hub safetensors  torch torchvision torchaudio torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 1.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cusparselt-cu13
    Found existing installation: nvidia-cusparselt-cu13 0.8.0
    Uninstalli

In [ ]:
# # 1. Clean out the potentially corrupted packages
# !pip uninstall torch torchvision torchaudio transformers accelerate -y

# # 2. Clear out pip's cached wheels to avoid re-installing a broken package
# !pip cache purge

# # 3. Install fresh, compatible versions
# !pip install torch torchvision --index-url https://pytorch.org
# !pip install -U accelerate "transformers[torch]"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# Import lib

In [ ]:
import os
import json, torch
import pandas as pd
import json
import os
from pathlib import Path
from dotenv import load_dotenv

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model




# Config & Path

In [ ]:
BASE_MODEL = "ntphuc149/ViLegalQwen2.5-1.5B-Base"
OUTPUT_DIR = "./legal-lora"
HF_REPO = "tmtdev/vn-legal-model"
HF_DATASET_REPO = "tmtdev/vn-legal-extraction"

MAX_LENGTH = 1024

DATA_DIR = "/content/drive/MyDrive/legal-ai/data"
DATA_PATH = os.path.join(DATA_DIR, "all_extract_citation_data.json")

IS_PUSH_DATASET_TO_HF = False

# Load data

In [ ]:
def load_json_file(file_path: str) -> list[dict]:
        """
        Load JSON data from a file.
        """
        if not Path(file_path).exists():
            print(f"File not found: {file_path}")
            raise FileNotFoundError(f"File not found: {file_path}")

        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return data


def eda_data(data: list[dict]):
  df = pd.DataFrame(data=data)

  display(df.head())
  display(df.info())
  # display(df.describe())


def split(legal_data: list[dict], test_size: float = 0.2):
  print("Cấu trúc 1 dòng mẫu:", legal_data[0])

  require_columns = ["query_text", "question_text", "target"]

  data = Dataset.from_list(legal_data).filter(
      lambda x: all(x.get(k) for k in ["query_text", "question_text", "target"])
  )
  print("Số dòng sau khi lọc:", len(data))

  splits = data.train_test_split(test_size=test_size, seed=42)
  temp = splits["test"].train_test_split(test_size=0.5, seed=42)

  train_data = splits["train"]
  val_data = temp["train"]
  test_data = temp["test"]

  print(len(train_data), len(val_data), len(test_data))

  return train_data, val_data, test_data


# Enable push
if IS_PUSH_DATASET_TO_HF:
  # Load
  legal_data = load_json_file(DATA_PATH)
  print(len(legal_data))

  # EDA
  eda_data(legal_data)

  # Split
  train_data, val_data, test_data = split(legal_data)



# Push to HF


# Load HF





# Push data Huggingface

In [ ]:
from huggingface_hub import login, create_repo, HfApi
from huggingface_hub.utils import validate_repo_id
from huggingface_hub import notebook_login
from datasets import DatasetDict
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Or use 'huggingface-cli login' in the terminal
notebook_login()
# login()

api = HfApi()
print(api.whoami())

if IS_PUSH_DATASET_TO_HF:
  dataset = DatasetDict({
      "train": train_data,
      "validation": val_data,
      "test": test_data
  })

  # Push dataset
  dataset.push_to_hub(
      repo_id=HF_DATASET_REPO,
      private=False
  )
else:
  print("NOT PUSH")



{'type': 'user', 'id': '6aa4f8abe4f23f13c474cdea', 'name': 'tmtdev', 'fullname': 'Minh Thuyên Trần', 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/6aa4f8abe4f23f13c474cdea/SoF8zS_qzs2sVtlzaO88D.jpeg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'legal-token', 'role': 'fineGrained', 'createdAt': '2026-10-12T06:15:03.732Z', 'fineGrained': {'canReadGatedRepos': True, 'global': ['discussion.write'], 'scoped': [{'entity': {'_id': '6acc6ed68e680ecfa7dfb03b', 'type': 'dataset', 'authorId': '6aa4f8abe4f23f13c474cdea', 'name': 'tmtdev/vn-legal-extraction'}, 'permissions': ['repo.content.read', 'repo.access.read', 'discussion.write', 'repo.write']}, {'entity': {'_id': '6aa4f8abe4f23f13c474cdea', 'type': 'user', 'name': 'tmtdev'}, 'permissions': ['repo.content.read', 'repo.access.read', 'repo.contribute.write', 'repo.write', 'inference.serverless.write', 'inference.endpoints.infer.write', 'inference.endpoints.write', 'user.w

# Load data huggingface

In [ ]:
from datasets import load_dataset

if not IS_PUSH_DATASET_TO_HF:
    dataset = load_dataset(HF_DATASET_REPO)
    print(dataset)
    train_data = dataset["train"]
    val_data = dataset["validation"]
    test_data = dataset["test"]

README.md:   0%|          | 0.00/882 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  499kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 78.4kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8438 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1055 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1055 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['query_text', 'question_text', 'target'],
        num_rows: 8438
    })
    validation: Dataset({
        features: ['query_text', 'question_text', 'target'],
        num_rows: 1055
    })
    test: Dataset({
        features: ['query_text', 'question_text', 'target'],
        num_rows: 1055
    })
})


# Process message template

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side = "right"

SYSTEM_PROMPT = """
Bạn là trợ lý xử lý truy vấn pháp luật Việt Nam.
Dựa vào dữ liệu đã học, hãy tạo JSON hợp lệ gồm:
query_text, question_text, context.
Không thêm nội dung ngoài JSON.
"""

config.json:   0%|          | 0.00/708 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.26k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

In [ ]:
def preprocess(example):
    target = json.dumps({
        "query_text": example["query_text"],
        "question_text": example["question_text"],
        "target": example["target"]
    }, ensure_ascii=False)

    prompt = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["query_text"]}
    ]

    full = prompt + [{"role": "assistant", "content": target}]

    prompt_result = tokenizer.apply_chat_template(
        prompt,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True
    )

    full_result = tokenizer.apply_chat_template(
        full,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True
    )

    prompt_ids = prompt_result["input_ids"]
    full_ids = full_result["input_ids"]

    # Một số trường hợp trả về dict/BatchEncoding
    if isinstance(prompt_ids, dict):
        prompt_ids = prompt_ids["input_ids"]

    if isinstance(full_ids, dict):
        full_ids = full_ids["input_ids"]

    prompt_ids = list(prompt_ids)
    full_ids = list(full_ids)

    full_ids = full_ids[:MAX_LENGTH]
    prompt_len = min(len(prompt_ids), len(full_ids))


    """
    • Trong thư viện PyTorch, hàm tính toán độ lỗi phổ biến là CrossEntropyLoss cấu hình mặc định tham số ignore_index = -100.
    Khi hàm này thấy nhãn có giá trị là -100, nó sẽ bỏ qua tự động và không tính điểm lỗi cho token đó.
    1. [-100] * prompt_len: Tạo ra một danh sách chứa toàn các số -100, với độ dài bằng đúng số lượng từ/token của phần câu lệnh (prompt_len).
    Điều này đảm bảo toàn bộ phần câu lệnh sẽ không bị tính loss.
    2. +: Ghép nối hai danh sách lại với nhau.
    3. full_ids[prompt_len:]: Lấy các ID token thực tế của phần câu trả lời (phần còn lại sau câu lệnh).
    Đây là phần mà mô hình bắt buộc phải học để dự đoán chính xác, nên giữ nguyên ID gốc của từ thay vì thay bằng -100.
    """
    labels = [-100] * prompt_len + full_ids[prompt_len:]

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels
    }

train_data = train_data.map(
    preprocess,
    remove_columns=train_data.column_names
)

val_data = val_data.map(
    preprocess,
    remove_columns=val_data.column_names
)

test_data = test_data.map(
    preprocess,
    remove_columns=test_data.column_names
)

Map:   0%|          | 0/8438 [00:00<?, ? examples/s]

Map:   0%|          | 0/1055 [00:00<?, ? examples/s]

Map:   0%|          | 0/1055 [00:00<?, ? examples/s]

In [ ]:
result = tokenizer.apply_chat_template(
    [{"role": "user", "content": "test"}],
    tokenize=True,
    add_generation_prompt=True
)

print(type(result))
print(result.keys() if hasattr(result, "keys") else type(result))

<class 'transformers.tokenization_utils_base.BatchEncoding'>
KeysView({'input_ids': [151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 1944, 151645, 198, 151644, 77091, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]})


In [ ]:
print(train_data.column_names)
print(train_data)
print(type(train_data[0]["input_ids"]))
print(train_data[0]["input_ids"][:5])

['input_ids', 'attention_mask', 'labels']
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 8438
})
<class 'list'>
[151644, 8948, 271, 94917, 37915]


In [ ]:
!pip install --upgrade torchao


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 19.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

model = get_peft_model(model, LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    task_type="CAUSAL_LM"
))

model.print_trainable_parameters()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

W1012 08:53:09.916000 637 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        label_pad_token_id=-100
    )
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.058729,0.059667


TrainOutput(global_step=528, training_loss=0.06407022227843602, metrics={'train_runtime': 2528.8306, 'train_samples_per_second': 3.337, 'train_steps_per_second': 0.209, 'total_flos': 1.6611459788034048e+16, 'train_loss': 0.06407022227843602, 'epoch': 1.0})

In [ ]:
print("Validation:", trainer.evaluate(val_data))
print("Test:", trainer.evaluate(test_data))

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Training Loss,Validation Loss,Epoch
0.058729,0.059667,1


Validation: {'eval_loss': 0.05966705456376076}


Training Loss,Validation Loss,Epoch
0.058729,0.059902,1


Test: {'eval_loss': 0.05990187078714371}


('./legal-lora/tokenizer_config.json',
 './legal-lora/chat_template.jinja',
 './legal-lora/tokenizer.json')

In [ ]:
# Validation và test loss
val_metrics = trainer.evaluate(
    eval_dataset=val_data,
    metric_key_prefix="val"
)

test_metrics = trainer.evaluate(
    eval_dataset=test_data,
    metric_key_prefix="test"
)

print("Validation:", val_metrics)
print("Test:", test_metrics)

In [ ]:
from huggingface_hub import login
login()

model.push_to_hub(HF_REPO, private=True)
tokenizer.push_to_hub(HF_REPO, private=True)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp9e8ovxx2/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/tmtdev/vn-legal-model/commit/0939ba47595aa39278ff2894ba71c7d88cac0be2', commit_message='Upload tokenizer', commit_description='', oid='0939ba47595aa39278ff2894ba71c7d88cac0be2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tmtdev/vn-legal-model', endpoint='https://huggingface.co', repo_type='model', repo_id='tmtdev/vn-legal-model'), pr_revision=None, pr_num=None)

In [ ]:
import json
import torch

def predict(query_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query_text}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    # Qwen dùng <|im_end|> để kết thúc một lượt hội thoại
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

    stop_ids = [tokenizer.eos_token_id]
    if im_end_id is not None and im_end_id not in stop_ids:
        stop_ids.append(im_end_id)

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            eos_token_id=stop_ids,
            pad_token_id=tokenizer.pad_token_id
                if tokenizer.pad_token_id is not None
                else tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )

    input_length = inputs["input_ids"].shape[-1]

    result = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()

    # Nếu model sinh thêm nội dung sau JSON, chỉ lấy object JSON đầu tiên
    try:
        decoder = json.JSONDecoder()
        start = result.find("{")

        if start == -1:
            return {"error": "Model không sinh JSON hợp lệ", "raw_output": result}

        parsed, _ = decoder.raw_decode(result[start:])
        return parsed

    except json.JSONDecodeError:
        return {"error": "Không thể parse JSON", "raw_output": result}


result = predict(
    "Theo khoản 2 Điều 13 Nghị định 10/2020/NĐ-CP, thời gian lưu trữ hình ảnh là bao lâu?"
)

print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "query_text": "Theo khoản 2 Điều 13 Nghị định 10/2020/NĐ-CP, thời gian lưu trữ hình ảnh là bao lâu?",
  "question_text": "Nghị định 10/2020/NĐ-CP, thời gian lưu trữ hình ảnh là bao lâu?",
  "target": {
    "part": null,
    "chapter": null,
    "section": null,
    "subsection": null,
    "article": "13",
    "clause": "2",
    "point": null,
    "document": "Nghị định 10/2020/NĐ-CP"
  }
}


In [ ]:
# run with test

test = [
        {
        "query_text": "Chương I của Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013 quy định về vấn đề gì?",
        "question_text": "Chương I của  quy định về vấn đề gì?",
        # "target": {
        #     "part": null,
        #     "chapter": "I",
        #     "section": null,
        #     "subsection": null,
        #     "article": null,
        #     "clause": null,
        #     "point": null,
        #     "document": "Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013"
        # }
    },
    {
        "query_text": "Nội dung Điều 1 theo Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013 là gì?",
        "question_text": "Nội dung Điều 1 theo  là gì?",
        # "target": {
        #     "part": null,
        #     "chapter": null,
        #     "section": null,
        #     "subsection": null,
        #     "article": "1",
        #     "clause": null,
        #     "point": null,
        #     "document": "Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013"
        # }
    },
]


for item in test:
  result = predict(item["query_text"])
  print(json.dumps(result, ensure_ascii=False, indent=2))


{
  "query_text": "Chương I của Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013 quy định về vấn đề gì?",
  "question_text": "Chương I của  quy định về vấn đề gì?",
  "target": {
    "part": null,
    "chapter": "I",
    "section": null,
    "subsection": null,
    "article": null,
    "clause": null,
    "point": null,
    "document": "Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013"
  }
}
{
  "query_text": "Nội dung Điều 1 theo Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013 là gì?",
  "question_text": "Nội dung Điều 1 theo  là gì?",
  "target": {
    "part": null,
    "chapter": null,
    "section": null,
    "subsection": null,
    "article": "1",
    "clause": null,
    "point": null,
    "document": "Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013"
  }
}
